# Phase de Modélisation (Pipeline et Entraînement)
Ce notebook contient la création du Data Generator (Phase 3) et l'entraînement des modèles de segmentation sémantique (Phase 4).


## Contexte et Démarche Méthodologique

Ce notebook constitue la réponse technique de conception de modèles de l'Intelligence Artificielle de vision.
Pour exposer pleinement notre démarche d'Industrialisation MLOps aux non-initiés, voici les choix que nous avons encodés dans les prochaines cellules :

### 1. Continuité (Les Variables de Référence)
Le pipeline se base bien évidemment sur l'analyse de données (EDA). Notre dictionnaire (*LABEL_TO_CATEGORY*) capable de traduire mathématiquement 34 méga-classes d'entrée vers nos 8 classes visées de sorties est incorporé et servira de traducteur à la volée.

### 2. L'Optimisation : Générateur de Données natif TensorFlow (Keras Sequence)
La pierre angulaire de ce projet est le constructeur `CityscapesGenerator`. Il n'est pas possible de stocker 20 Gigaoctets de photos temporaires dans la RAM.
Il offre donc : 
*   Une lecture des Images/Masques couplée et répartie en **Batches** (petits paquets) pour l'entraînement (entrainement itératif).
*   Un **Redimensionnement dynamique** (256x512 ou autre) au vol.
*   Il applique une interpolation `Nearest-Neighbor` strictement rigoureuse sur les masques (qui ne supportent absolument pas le lissage pixel par pixel dû au passage vectoriel de classes distinctes).
*   Un encodage multidimensionnel via `to_categorical` : Une matrice **One-Hot Encoded** indispensable pour utiliser un calcul d'entropie croisée final.

### 3. La Régularisation : Data Augmentation via la librairie spécialisée Albumentations
L'un des défis classiques est le surapprentissage. Comment forcer le modèle à généraliser intelligemment ? Nous ne pouvons pas le nourrir toujours du même lot à l'identique. 
L'augmentation intègre, par exemple, des reflets optiques (Horizontal Flip) mais doit ABSOLUMENT se synchroniser avec le masque sémantique qui l'accompagne.

*(La suite de ce Notebook concernera la génération des blocs architecturaux, ex: U-Net et sa validation)*

In [1]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import layers, models, utils
from PIL import Image
import albumentations as A
import time


## 1. Constants et Importation du Mapping
Nous reprenons la logique initiée en Phase 1. On intègre le mapping pour réduire dynamiquement nos 34 classes vers les 8 cibles d'entraînement, et on prépare les chemins d'accès vers nos dossiers de données.


In [2]:
LABEL_TO_CATEGORY = {
    0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0,   # void
    7: 1, 8: 1, 9: 1, 10: 1,                    # flat
    11: 2, 12: 2, 13: 2, 14: 2, 15: 2, 16: 2,   # construction
    17: 3, 18: 3, 19: 3, 20: 3,                 # object
    21: 4, 22: 4,                               # nature
    23: 5,                                      # sky
    24: 6, 25: 6,                               # human
    26: 7, 27: 7, 28: 7, 29: 7, 30: 7, 31: 7, 32: 7, 33: 7  # vehicle
}

TRAIN_IMG_DIR = 'data/P8_Cityscapes_leftImg8bit_trainvaltest/leftImg8bit/train/'
TRAIN_MASK_DIR = 'data/P8_Cityscapes_gtFine_trainvaltest/gtFine/train/'

VAL_IMG_DIR = 'data/P8_Cityscapes_leftImg8bit_trainvaltest/leftImg8bit/val/'
VAL_MASK_DIR = 'data/P8_Cityscapes_gtFine_trainvaltest/gtFine/val/'


## 2. Création du Data Generator (Keras Sequence)
Afin de ne pas surcharger la RAM et le GPU, nous allons implémenter `tf.keras.utils.Sequence`. Ce générateur agira "à la volée" pendant l'entraînement :
- Il charge une série d'images (batchs).
- Redimensionne.
- Applique l'augmentation d'image de façon identique sur le masque et la photo.
- Remappe les classes (- de 32 à 8).
- Et transforme la couche de segment via le système "One-Hot Encoding" pour croiser avec les probabilités Softmax de notre futur réseau de Neurones.


In [3]:
class CityscapesGenerator(utils.Sequence):
    def __init__(self, image_dir, mask_dir, batch_size=8, img_size=(256, 512), augment=False):
        # On liste toutes les images et masks valides par dossier avec la lib glob.
        self.image_paths = sorted(glob.glob(os.path.join(image_dir, '**/*_leftImg8bit.png'), recursive=True))
        self.mask_paths = sorted(glob.glob(os.path.join(mask_dir, '**/*_gtFine_labelIds.png'), recursive=True))
        
        assert len(self.image_paths) == len(self.mask_paths), "Erreur : Les nombres d'images et de masques divergent."
        
        self.batch_size = batch_size
        self.img_size = img_size
        self.augment = augment
        
        # Optionnel pour l'entraînement : définition de transformations symétriques (albumentations)
        if self.augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
            ])
            
    def __len__(self):
        # Définit le nombre total de "steps" ou de "lots" pour une seule EPOCH !
        return int(np.floor(len(self.image_paths) / self.batch_size))
        
    def __getitem__(self, index):
        # Appelé par les API de fit() de KERAS pour récolter un Lot précis via son Index.
        batch_img_paths = self.image_paths[index * self.batch_size:(index + 1) * self.batch_size]
        batch_mask_paths = self.mask_paths[index * self.batch_size:(index + 1) * self.batch_size]
        
        # 1. Remplir des array NumPy (plus facile & rapide que les list.append())
        X = np.empty((self.batch_size, self.img_size[0], self.img_size[1], 3), dtype=np.float32)
        y = np.empty((self.batch_size, self.img_size[0], self.img_size[1]), dtype=np.uint8)
        
        for i, (img_path, mask_path) in enumerate(zip(batch_img_paths, batch_mask_paths)):
            # Lecture
            img = np.array(Image.open(img_path).resize((self.img_size[1], self.img_size[0])))
            
            # Lecture fine du label et interpolation "proche voisin" indispensable (ne jamais faire de bilinéaire sur des index de catégories!)
            mask = np.array(Image.open(mask_path).resize((self.img_size[1], self.img_size[0]), resample=Image.NEAREST))
            
            # Remappage 34 -> 8 (Vectorisation via liste préexistante !)
            mask_8_cat = np.zeros_like(mask)
            for original_id, mapped_id in LABEL_TO_CATEGORY.items():
                mask_8_cat[mask == original_id] = mapped_id
                
            # 2. Application de l'Augmentation d'image de la lib Albumentation synchronisée !
            if self.augment:
                augmented = self.transform(image=img, mask=mask_8_cat)
                img = augmented['image']
                mask_8_cat = augmented['mask']
                
            # 3. Assignation au lot (X <- Float normalisé [0-1], y <- labels terrain entier entre 0 & 7)
            X[i,] = img / 255.0
            y[i,] = mask_8_cat
            
        # IMPORTANT : Pour une classification multi-classe, utiliser Categorical Cross Entropy requiert de One Hote Encoded vos Masks (Couches)
        Y = tf.keras.utils.to_categorical(y, num_classes=8)
        return X, Y


## 3. Initialisation et validation du générateur
Créons notre jeu de test pour visualiser la sortie d'un batch et s'assurer que notre Tensor Flow digère bien son input.

In [4]:
# Instanciation pour l'entraînement
train_gen = CityscapesGenerator(TRAIN_IMG_DIR, TRAIN_MASK_DIR, batch_size=4, img_size=(256, 512), augment=True)
val_gen = CityscapesGenerator(VAL_IMG_DIR, VAL_MASK_DIR, batch_size=4, img_size=(256, 512), augment=False)

# Récupération d'un exemple du premier Batch du générateur "train_gen" :
X_batch, Y_batch = train_gen[0]
print("Shape Image (X) :", X_batch.shape)
print("Shape Mask One-Hot (Y) :", Y_batch.shape)

# Afin d'afficher nos Y, on reverse notre One-Hot Encoder avec ArgMax
y_labels_batch = np.argmax(Y_batch, axis=-1)
print("Shape Mask décodé :", y_labels_batch.shape)


Shape Image (X) : (4, 256, 512, 3)
Shape Mask One-Hot (Y) : (4, 256, 512, 8)
Shape Mask décodé : (4, 256, 512)


## 4. Modèle Baseline : L'Architecture U-Net

Pour valider toute la chaîne des données (de l'image disque à la prédiction), il faut commencer par un modèle "Baseline" simple, sans aucun module pré-entrainé complexe. 

Ici nous codons un **U-Net** natif Keras.
Cette topologie très classique en vision par ordinateur possède deux parties en "U" :
1. **Un Encodeur (Descente) :** Il extrait les concepts (textures, formes) via les convolutions, tout en écrasant la taille (Pooling) pour gagner en compréhension globale.
2. **Un Décodeur (Montée) :** Il ré-augmente l'image (UpSampling) pour recréer la carte finale de classe pixel-à-pixel, aidé par les *Skip Connections* (fleches horizontales) qui rapatrient les détails fins d'origine pour ne pas avoir un résultat trop pixelisé.

In [5]:
def build_unet(input_shape=(256, 512, 3), num_classes=8):
    inputs = layers.Input(shape=input_shape)
    
    # --- ENCODEUR (Descente) ---
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)

    # --- BOTTLENECK (Partie Centrale) ---
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c4)

    # --- DÉCODEUR (Remontée avec liaisons "Skip Connections") ---
    u5 = layers.UpSampling2D((2, 2))(c4)
    u5 = layers.concatenate([u5, c3]) # Fusion avec les données fines du bloc 3 horizontal
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u5)
    c5 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c5)

    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.concatenate([u6, c2]) # Fusion bloc 2
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c6)

    u7 = layers.UpSampling2D((2, 2))(c6)
    u7 = layers.concatenate([u7, c1]) # Fusion bloc 1
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(c7)

    # --- SORTIE ---
    # Convolution finale qui fait une matrice de (256, 512, 8).
    # L'activation Softmax transformera les canaux en probabilités (0% à 100% que tel pixel soit une route).
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(c7)

    model = models.Model(inputs=[inputs], outputs=[outputs])
    return model

model_unet = build_unet((256, 512, 3), 8)
model_unet.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 512,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 512,  │      9,248 │ conv2d[0][0]      │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 256,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 256,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 256,  │     36,928 │ conv2d_2[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 128,   │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 128,   │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 128,   │    147,584 │ conv2d_4[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 64,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 64,    │    295,168 │ max_pooling2d_2[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 64,    │    590,080 │ conv2d_6[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 64, 128,   │          0 │ conv2d_7[0][0]    │
│ (UpSampling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64, 128,   │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 384)              │            │ conv2d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 64, 128,   │    442,496 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 64, 128,   │    147,584 │ conv2d_8[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 128, 256,  │          0 │ conv2d_9[0][0]  

 Total params: 1,947,112 (7.43 MB)

 Trainable params: 1,947,112 (7.43 MB)

 Non-trainable params: 0 (0.00 B)

## 5. Compilation, Callbacks et Lancement (Phase 4)

Avant toute itération avec TensorFlow, 3 configurations MLOps s'imposent :
1. **Les Métriques :** Nous choisissons la Mean IoU car l'accuracy classique donnerait l'impression de succès alors que les petits objets (vélos) passeraient inaperçus derrière la proportion du ciel.
2. **La Loss (Fonction d'Erreur) :** Pour un Baseline simple, l'entropie croisée catégorielle, souvent combinée à la technique de matrices *One-Hot encoded* du générateur fait l'affaire.
3. **Les Callbacks :** Véritables "anges gardiens" du code, garantissant sauvegarde automatique (`ModelCheckpoint`), abandon face au sur-entraînement (`EarlyStopping`) et régulation mathématique du pas d'apprentissage (`ReduceLROnPlateau`).

In [6]:
# Paramétrage des Métriques
metrics = ['accuracy', tf.keras.metrics.MeanIoU(num_classes=8, name='mean_iou')]

# Compilation avec un optimisateur Adam
model_unet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=metrics)

# Callbacks indispensables en milieu professionnel
callbacks_list = [
    # Sauvegarde constante du strict meilleur modèle en valid_iou pour éviter de bruler le disque
    tf.keras.callbacks.ModelCheckpoint('best_unet_model.keras', save_best_only=True, monitor='val_mean_iou', mode='max', verbose=1),
    # Stoppe le système s'il s'enferme dans du sur-apprentissage (10 Epochs sans progression globale)
    tf.keras.callbacks.EarlyStopping(patience=10, monitor='val_mean_iou', mode='max', restore_best_weights=True),
    # Divise subtilement le pas d'apprentissage par 2 lorsque l'IA patine 
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, monitor='val_mean_iou', mode='max', min_lr=1e-6, verbose=1)
]

print("✅ Réseau U-Net compilé ! Prêt pour le décollage.")


history = model_unet.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=20,
    callbacks=callbacks_list
)


✅ Réseau U-Net compilé ! Prêt pour le décollage.


/Users/j/Documents/OC_Inge_IA/Projet_8/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.4744 - loss: 1.4216 - mean_iou: 0.4630
Epoch 1: val_mean_iou improved from -inf to 0.43750, saving model to best_unet_model.keras
743/743 ━━━━━━━━━━━━━━━━━━━━ 1480s 2s/step - accuracy: 0.4745 - loss: 1.4213 - mean_iou: 0.4642 - val_accuracy: 0.6979 - val_loss: 0.8599 - val_mean_iou: 0.4375 - learning_rate: 1.0000e-04
Epoch 2/20
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7179 - loss: 0.8259 - mean_iou: 0.4630
Epoch 2: val_mean_iou did not improve from 0.43750
743/743 ━━━━━━━━━━━━━━━━━━━━ 1484s 2s/step - accuracy: 0.7179 - loss: 0.8258 - mean_iou: 0.4642 - val_accuracy: 0.7673 - val_loss: 0.7095 - val_mean_iou: 0.4375 - learning_rate: 1.0000e-04
Epoch 3/20
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7845 - loss: 0.6665 - mean_iou: 0.4630
Epoch 3: val_mean_iou did not improve from 0.43750
743/743 ━━━━━━━━━━━━━━━━━━━━ 1477s 2s/step - accuracy: 0.7845 - loss: 0.6665 - mean_iou: 0.4642 - val_accuracy: 0.792

## 6. Modèle Avancé : U-Net avec Transfer Learning (MobileNetV2)

Afin de fournir une comparaison de performances (Phase 4.2), nous allons implémenter un deuxième réseau de neurones. Plutôt que de repartir de zéro comme avec le Baseline, il est souvent très pertinent d'utiliser le **Transfer Learning**.

Cette technique consiste à remplacer la partie Encodeur de notre U-Net par un grand réseau existant pré-entraîné sur des millions d'images (ici *MobileNetV2* entrainé sur *ImageNet*). L'avantage ?
- MobileNetV2 sait déjà reconnaitre de manière experte ce qu'est une ligne droite, un contour, diverses textures, puisqu'il a déjà appris sur des millions de photos classiques.
- Son architecture est mathématiquement optimisée pour les terminaux basse consommation (smartphones, IoT embarqué,... ce qui répond logiquement au besoin d'intégration à bord des futurs véhicules de *Future Vision Transport*).

Dans un premier temps, les couches récupérées seront **gelées** (trainable = False) pour ne forcer l'apprentissage que sur la partie du Décodeur que nous rajoutons nous-même derrière.

In [7]:
def build_unet_mobilenet(input_shape=(256, 512, 3), num_classes=8):
    # 1. ENCODEUR : L'Import du Transfer Learning
    # On importe le réseau entier sans la tête de classification finale
    base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    
    # Congeler les poids existants (facilite la convergence au début)
    base_model.trainable = False

    # MobileNetV2 réduit mathématiquement l'image. Pour faire des Skip Connections propres, on repère 5 étages de sortie distincts par leur nom (propres à TensorFlow)
    layer_names = [
        'block_1_expand_relu',   # Extrait à la taille 128x256
        'block_3_expand_relu',   # Extrait à 64x128
        'block_6_expand_relu',   # Extrait à 32x64
        'block_13_expand_relu',  # Extrait à 16x32
        'block_16_project',      # Bottleneck central à 8x16
    ]
    base_model_outputs = [base_model.get_layer(name).output for name in layer_names]
    
    # Un sous-modèle qui agit uniquement comme un extracteur de Features multi-échelles pour nous faciliter la vie
    down_stack = tf.keras.Model(inputs=base_model.input, outputs=base_model_outputs)

    # --- DÉBUT DU GRAPH ---
    inputs = tf.keras.layers.Input(shape=input_shape)
    
    # Passage de l'image vierge à travers l'encodeur puissant
    skips = down_stack(inputs)
    x = skips[-1] # On prend le fond très condensé (8x16) comme base
    skips = reversed(skips[:-1]) # On inverse l'ordre des autres extraits pour notre remontée future

    # 2. DÉCODEUR : La Remontée en U avec nos connexions horizontales (Skip-Connections)
    decoder_filters = [512, 256, 128, 64]
    
    for skip, filters in zip(skips, decoder_filters):
        x = tf.keras.layers.UpSampling2D((2, 2))(x)
        x = tf.keras.layers.concatenate([x, skip])
        x = tf.keras.layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
        x = tf.keras.layers.Conv2D(filters, (3, 3), activation='relu', padding='same')(x)

    # Ultime remontée pour récupérer exactement la taille initiale (256x512)
    x = tf.keras.layers.UpSampling2D((2, 2))(x)
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)

    # Convolution finale projetée sur 8 canaux (Nos 8 catégories Cityscapes)
    outputs = tf.keras.layers.Conv2D(num_classes, (1, 1), activation='softmax')(x)

    return tf.keras.Model(inputs=inputs, outputs=outputs)

# Instanciation et Analyse de la quantité de paramètres entraînables (très faible !)
model_transfer = build_unet_mobilenet((256, 512, 3), 8)
model_transfer.summary()


/var/folders/dm/pvk529ls4pg3bshgj9btlsc80000gn/T/ipykernel_22928/4247413247.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 256, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ [(None, 128, 256, │  1,841,984 │ input_layer_2[0]… │
│ (Functional)        │ 96), (None, 64,   │            │                   │
│                     │ 128, 144), (None, │            │                   │
│                     │ 32, 64, 192),     │            │                   │
│                     │ (None, 16, 32,    │            │                   │
│                     │ 576), (None, 8,   │            │                   │
│                     │ 16, 320)]         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_3     │ (None, 16, 32,    │          0 │ functional_1[0][… │
│ (UpSampling2D)      │ 320)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 16, 32,    │          0 │ up_sampling2d_3[… │
│ (Concatenate)       │ 896)              │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 16, 32,    │  4,129,280 │ concatenate_3[0]… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 16, 32,    │  2,359,808 │ conv2d_15[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_4     │ (None, 32, 64,    │          0 │ conv2d_16[0][0]   │
│ (UpSampling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 32, 64,    │          0 │ up_sampling2d_4[… │
│ (Concatenate)       │ 704)              │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 32, 64,    │  1,622,272 │ concatenate_4[0]… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 32, 64,    │    590,080 │ conv2d_17[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_5     │ (None, 64, 128,   │          0 │ conv2d_18[0][0]   │
│ (UpSampling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 64, 128,   │          0 │ up_sampling2d_5[… │
│ (Concatenate)       │ 400)              │            │ functional_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 64, 128,   │    460,928 │ concatenate_5[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 64, 128,   │    147,584 │ conv2d_19[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_6     │ (None, 128, 256,  │          0 │ conv2d_20[0][0]   │
│ (UpSampling2D)      │ 128)              │            │                 

 Total params: 11,345,928 (43.28 MB)

 Trainable params: 9,503,944 (36.25 MB)

 Non-trainable params: 1,841,984 (7.03 MB)

### Lancement de l'Entraînement TLAB

Ce puissant modèle s'entraîne exactement de la même manière (et bénéficiera de ce fait de notre super séquence `train_gen` codée précédemment) !
On configure un nom de fichier différent pour le callback afin de stocker les 2 réseaux sans concurrence.

In [8]:
# Paramétrage des Métriques inchangé
metrics = ['accuracy', tf.keras.metrics.MeanIoU(num_classes=8, name='mean_iou')]

model_transfer.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy',
              metrics=metrics)

# Callbacks (Nom de sauvegarde Keras mis à jour par prudence)
callbacks_list_transfer = [
    tf.keras.callbacks.ModelCheckpoint('best_mobilenet_unet_model.keras', save_best_only=True, monitor='val_mean_iou', mode='max', verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=8, monitor='val_mean_iou', mode='max', restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_mean_iou', mode='max', min_lr=1e-6, verbose=1)
]

print("✅ Réseau U-Net (Transfer Learning) compilé ! Prêt pour le décollage.")


history_transfer = model_transfer.fit(
    train_gen, 
    validation_data=val_gen, 
    epochs=25,
    callbacks=callbacks_list_transfer
)



✅ Réseau U-Net (Transfer Learning) compilé ! Prêt pour le décollage.
Epoch 1/25
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6858 - loss: 0.9962 - mean_iou: 0.4652
Epoch 1: val_mean_iou improved from -inf to 0.43832, saving model to best_mobilenet_unet_model.keras
743/743 ━━━━━━━━━━━━━━━━━━━━ 1434s 2s/step - accuracy: 0.6859 - loss: 0.9958 - mean_iou: 0.4664 - val_accuracy: 0.8468 - val_loss: 0.4778 - val_mean_iou: 0.4383 - learning_rate: 0.0010
Epoch 2/25
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8651 - loss: 0.4421 - mean_iou: 0.4669
Epoch 2: val_mean_iou improved from 0.43832 to 0.43889, saving model to best_mobilenet_unet_model.keras
743/743 ━━━━━━━━━━━━━━━━━━━━ 1436s 2s/step - accuracy: 0.8651 - loss: 0.4421 - mean_iou: 0.4681 - val_accuracy: 0.8643 - val_loss: 0.4366 - val_mean_iou: 0.4389 - learning_rate: 0.0010
Epoch 3/25
743/743 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8836 - loss: 0.3822 - mean_iou: 0.4707
Epoch 3: val_mean_iou improved from 0.43889 

In [9]:
stop_time = time.time()
temps_total = stop_time - start_time
print(f"Le temps total passé à calculer est {temps_total}")

NameError: name 'start_time' is not defined